In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score
import utils_ml
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.feature_selection import VarianceThreshold

In [2]:
X_arr = np.load('dataExt/features_filtered.npy', mmap_mode='r+')
sel = VarianceThreshold(threshold=(.9 * (1 - .9)))
X_arr = sel.fit_transform(X_arr)

np.save('dataExt/features_selected.npy',X_arr)

In [3]:
X_arr = np.load('dataExt/features_selected.npy', mmap_mode='r+')

In [4]:
# Load and concatenate all features
file_prefix = "dataExt/P"
file_suffix = "_intensity.pkl"
num_files = 35

class_to_poles = {
    0: [0, 0, 0],  # 1 pole on [bt]
    1: [1, 0, 0],  # 1 pole on [bb]
    2: [0, 1, 0],  # 1 pole on [tb]
    3: [0, 0, 1],  # 1 pole on [bt] and 1 pole on [bb]
    4: [2, 0, 0],  # 1 pole on [bb] and 1 pole on [tb]
    5: [0, 2, 0],  # 1 pole on each of [bt], [bb], and [tb]
    6: [0, 0, 2],  # 2 poles on [bb] and 1 pole on [tb]
    7: [1, 1, 0],   # 1 pole on [bb] and 2 poles on [tb]
    8: [1, 0, 1],
    9: [0, 1, 1],
    10: [3, 0, 0],
    11: [0, 3, 0],
    12: [0, 0, 3],
    13: [2, 1, 0],
    14: [2, 0, 1],
    15: [1, 2, 0],
    16: [0, 2, 1],
    17: [1, 0, 2],
    18: [0, 1, 2],
    19: [1, 1, 1],
    20: [4, 0, 0],
    21: [0, 4, 0],
    22: [0, 0, 4],
    23: [3, 1, 0],
    24: [3, 0, 1],
    25: [1, 3, 0],
    26: [0, 3, 1],
    27: [1, 0, 3],
    28: [0, 1, 3],
    29: [2, 2, 0],
    30: [2, 0, 2],
    31: [0, 2, 2],
    32: [2, 1, 1],
    33: [1, 2, 1],
    34: [1, 1, 2],
}


y_arr_classification = np.array([np.tile(i,np.load(f"{file_prefix}{0:02d}{file_suffix}", allow_pickle=True).shape[0]) for i in np.arange(num_files)]).flatten()
y_arr_regression = utils_ml.convert_labels(y_arr_classification, class_to_poles)*1.0

In [5]:
rg = CatBoostRegressor(
    verbose=100,
    loss_function="MultiRMSE" 
    )
X_train, X_val, y_train, y_val = train_test_split(X_arr, y_arr_regression, random_state=42, test_size=0.2)
rg.fit(X_train, y_train, eval_set=(X_val,y_val), plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 1.8582827	test: 1.8584192	best: 1.8584192 (0)	total: 514ms	remaining: 8m 33s
100:	learn: 0.8455210	test: 0.8442636	best: 0.8442636 (100)	total: 32.7s	remaining: 4m 51s
200:	learn: 0.7697442	test: 0.7691134	best: 0.7691134 (200)	total: 1m 3s	remaining: 4m 10s
300:	learn: 0.7382173	test: 0.7381899	best: 0.7381899 (300)	total: 1m 33s	remaining: 3m 37s
400:	learn: 0.7195676	test: 0.7200262	best: 0.7200262 (400)	total: 2m 3s	remaining: 3m 4s
500:	learn: 0.7052153	test: 0.7062459	best: 0.7062459 (500)	total: 2m 33s	remaining: 2m 32s
600:	learn: 0.6942153	test: 0.6958367	best: 0.6958367 (600)	total: 3m 3s	remaining: 2m 1s
700:	learn: 0.6853984	test: 0.6876137	best: 0.6876137 (700)	total: 3m 33s	remaining: 1m 30s
800:	learn: 0.6783462	test: 0.6811312	best: 0.6811312 (800)	total: 4m 2s	remaining: 1m
900:	learn: 0.6722018	test: 0.6755352	best: 0.6755352 (900)	total: 4m 32s	remaining: 29.9s
999:	learn: 0.6670729	test: 0.6709759	best: 0.6709759 (999)	total: 5m 1s	remaining: 0us

bestTest

In [6]:
accuracy_score(utils_ml.reconvert_labels(np.abs(np.round(rg.predict(X_val))), class_to_poles), utils_ml.reconvert_labels(y_val, class_to_poles))

0.5448

In [8]:
clf = CatBoostClassifier(
    verbose=100           # Print training progress every 100 iterations
)
X_train, X_val, y_train, y_val = train_test_split(X_arr, y_arr_classification,random_state=42, test_size=0.2)
clf.fit(X_arr, y_arr_classification, eval_set=(X_val,y_val),plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.121397
0:	learn: 2.7706416	test: 2.7683736	best: 2.7683736 (0)	total: 14s	remaining: 3h 52m 45s
100:	learn: 0.9332302	test: 0.9304178	best: 0.9304178 (100)	total: 20m 45s	remaining: 3h 4m 46s
200:	learn: 0.8331024	test: 0.8308019	best: 0.8308019 (200)	total: 41m 5s	remaining: 2h 43m 22s


KeyboardInterrupt: 

In [9]:
accuracy_score(clf.predict(X_val),y_val)

CatBoostError: There is no trained model to use predict(). Use fit() to train model. Then use this method.

In [22]:
# percentages = [0.5,0.75,0.9]
# num_features = X_train.shape[1]

# feature_importances = clf.get_feature_importance()
# feature_names = [f'Feature {i}' for i in range(X_train.shape[1])]

# # Rank features by importance
# ranked_features = sorted(zip(feature_names, feature_importances), key=lambda x: x[1], reverse=True)

# # Store results
# results = {}

# # Train and evaluate for each percentage
# for percent in percentages:
#     k = max(1, int(percent * num_features))  # Number of features to select
#     top_features = [name for name, importance in ranked_features[:k]]
#     selected_columns = [feature_names.index(name) for name in top_features]
    
#     # Reduce dataset
#     X_train_selected = X_train[:, selected_columns]
#     X_val_selected = X_val[:, selected_columns]
    
#     # Train with selected features
#     clf_reduced = CatBoostClassifier(verbose=100)
#     clf_reduced.fit(X_train_selected, y_train)
    
#     # Evaluate accuracy
#     y_pred = clf_reduced.predict(X_val_selected)
#     acc = accuracy_score(y_val, y_pred)
#     results[percent] = acc

# print("Accuracy with top features:")
# for percent, acc in results.items():
#     print(f"Top {int(percent * 100)}% features: {acc:.4f}")

In [21]:

# # Define a range of thresholds to test
# thresholds = np.arange(1, X_train.shape[1], 5)  # 50 thresholds between 0 and 0.5
# accuracies = []
# num_features_list = []

# best_accuracy = 0
# best_threshold = 0
# best_num_features = X_train.shape[1]

# for threshold in thresholds:
#     selector = SelectKBest(f_classif, k=int(threshold))
#     X_train_selected = selector.fit_transform(X_train, y_train)
#     X_test_selected = selector.transform(X_val)
    
#     # Initialize and fit a HistGradientBoostingClassifier with isotonic calibration
#     clf_reduced = CatBoostClassifier(verbose=100)
#     clf_reduced.fit(X_train_selected, y_train)
    
#     # Evaluate accuracy
#     y_pred = clf_reduced.predict(X_test_selected)
#     accuracy = accuracy_score(y_val, y_pred)
    
#     # Keep track of accuracy and number of features
#     num_features = X_train_selected.shape[1]
#     accuracies.append(accuracy)
#     num_features_list.append(num_features)
    
#     # Keep track of the best threshold based on accuracy and feature count
#     if accuracy > best_accuracy or (accuracy == best_accuracy and num_features < best_num_features):
#         best_accuracy = accuracy
#         best_threshold = threshold
#         best_num_features = num_features

# # Plot threshold vs number of features
# plt.figure(figsize=(10, 6))
# plt.plot(num_features_list, accuracies, label="Number of Features", color='blue', marker='o')
# plt.ylabel("Accuracy")
# plt.xlabel("Number of Features")
# plt.title("Threshold vs Number of Features")
# plt.grid(True)

# plt.show()

# # Print the best threshold and accuracy information
# print(f"Best Threshold: {best_threshold:.4f}")
# print(f"Best Accuracy: {best_accuracy:.4f}")
# print(f"Number of Features: {best_num_features}")
